In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]
STEMS = {
    "drums": "drums.wav",
    "vocals": "vocals.wav",
    "bass": "bass.wav",
    "other": "other.wav"
}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

# Data Exploration

In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):

    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS.values()} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS.values()} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)

        if not os.path.isdir(genre_path):
            print(f"[WARN] Genre folder missing: {genre}")
            continue

        valid_songs = []

        # ----------------- Scan all song folders -----------------
        for song in sorted(os.listdir(genre_path)):
            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            # ---- CHECK: completeness of stems only ----
            valid = True
            for stem_file in STEMS.values():
                stem_path = os.path.join(song_path, stem_file)
                if not os.path.exists(stem_path):
                    valid = False
                    break

            if valid:
                valid_songs.append(song_path)

        if len(valid_songs) == 0:
            print(f"[WARN] No valid songs found for genre: {genre}")
            continue

        # ----------------- Stratified Shuffle Split -----------------
        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * (1 - val_split))

        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        # ----------------- Populate dictionaries -----------------
        def add_to_dict(target_dict, song_list):
            for song_path in song_list:
                for stem_name, stem_file in STEMS.items():
                    target_dict[genre][stem_name].append(
                        os.path.join(song_path, stem_file)
                    )

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset

In [5]:
tr, val = build_dataset(DATA_ROOT)

In [6]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=30):
    records = []
    total_files = 0

    MIN_GAP_SEC = 0.1  # critical exam constraint

    for genre, stems in dataset_dict.items():
        for stem_name, file_list in stems.items():
            for file_path in file_list:
                total_files += 1
                try:
                    y, sr = librosa.load(file_path, sr=sr)
                    total_duration = len(y) / sr

                    intervals = librosa.effects.split(y, top_db=top_db)

                    silence_segments = []
                    silence_type = set()

                    # FULL SILENCE
                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type.add("full")

                    else:
                        # START
                        start_gap = intervals[0][0] / sr
                        if start_gap >= MIN_GAP_SEC:
                            silence_segments.append(start_gap)
                            silence_type.add("start")

                        # MIDDLE
                        for i in range(len(intervals) - 1):
                            gap = (intervals[i+1][0] - intervals[i][1]) / sr
                            if gap >= MIN_GAP_SEC:
                                silence_segments.append(gap)
                                silence_type.add("middle")

                        # END
                        end_gap = (len(y) - intervals[-1][1]) / sr
                        if end_gap >= MIN_GAP_SEC:
                            silence_segments.append(end_gap)
                            silence_type.add("end")

                        max_silence = max(silence_segments) if silence_segments else 0

                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ",".join(sorted(silence_type)),
                            "File_Path": file_path
                        })

                except:
                    continue

    print(f"[INFO] Total files scanned: {total_files}")
    return pd.DataFrame(records)

In [7]:
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
df_silence.shape

[INFO] Total files scanned: 3320


(680, 6)

In [8]:
df_silence.head()

,Genre,Stem,Duration,Max_Silence_Sec,Silence_Location,File_Path
0,blues,drums,30.01,16.28,"end,middle,start",/kaggle/input/competitions/jan-2026-dl-gen-ai-...
1,blues,drums,30.01,12.35,"middle,start",/kaggle/input/competitions/jan-2026-dl-gen-ai-...
2,blues,drums,30.01,11.26,"end,middle,start",/kaggle/input/competitions/jan-2026-dl-gen-ai-...
3,blues,drums,30.01,9.57,"end,middle",/kaggle/input/competitions/jan-2026-dl-gen-ai-...
4,blues,drums,30.01,10.89,"end,middle,start",/kaggle/input/competitions/jan-2026-dl-gen-ai-...


In [9]:
stems_audio = []
try:
    for key in STEM_KEYS:
        # Get stem list for selected genre
        stem_files = tr[GENRE_TO_TEST][key]

        # Select song by index
        file_path = stem_files[SONG_INDEX]

        # Load audio (Duration 5.0s for speed/consistency)
        y, sr = librosa.load(file_path, sr=SR, duration=5.0)

        stems_audio.append(y)

    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [10]:
# ------------------- write your code here -------------------------------

# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.stack(stems_audio, axis=0)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw ** 2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#-------------------------------------------------------------------------

## Milestone 1

In [11]:
LESS_THAN_4KB = 0
LESS_THAN_50491_MB = 0

SIZE_4KB = 4 * 1024
SIZE_50491_MB = 5.0491 * 1024 * 1024

for genre in GENRES:
    genre_path = os.path.join(DATA_ROOT, genre)
    if not os.path.isdir(genre_path):
        continue

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue

        for stem_file in STEMS.values():
            stem_path = os.path.join(song_path, stem_file)
            if not os.path.exists(stem_path):
                continue

            size = os.path.getsize(stem_path)

            if size < SIZE_4KB:
                LESS_THAN_4KB += 1

            if size < SIZE_50491_MB:
                LESS_THAN_50491_MB += 1

Q1_ANSWER = LESS_THAN_4KB + LESS_THAN_50491_MB
print("q1 ",Q1_ANSWER)

GREATER_THAN_50493_MB = 0
LESS_THAN_50491_MB = 0

SIZE_50493_MB = 5.0493 * 1024 * 1024
SIZE_50491_MB = 5.0491 * 1024 * 1024

for genre in GENRES:
    genre_path = os.path.join(DATA_ROOT, genre)
    if not os.path.isdir(genre_path):
        continue

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue

        for stem_file in STEMS.values():
            stem_path = os.path.join(song_path, stem_file)
            if not os.path.exists(stem_path):
                continue

            size = os.path.getsize(stem_path)

            if size > SIZE_50493_MB:
                GREATER_THAN_50493_MB += 1

            if size < SIZE_50491_MB:
                LESS_THAN_50491_MB += 1

Q2_ANSWER = abs(GREATER_THAN_50493_MB - LESS_THAN_50491_MB)
print("q2 ",Q2_ANSWER)


train_reggae_drums = len(tr["reggae"]["drums"])
val_country_vocals = len(val["country"]["vocals"])

Q3_ANSWER = abs(train_reggae_drums - val_country_vocals)
print("q3 ",Q3_ANSWER)


q4=df_silence.shape[0]
print("q4 ",q4)


q5 = df_silence[
    df_silence["Stem"] == "vocals"
].shape[0]
print("q5 ",q5)


q6 = df_silence[
    df_silence["Stem"] == "vocals"
]["Max_Silence_Sec"].mean()
print("q6 ",q6)


q7 = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
].shape[0]
print("q7 ",q7)


q8 = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle")
].shape[0]
print("q8 ",q8)


q9 = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0]
print("q9 ",q9)


print("q11 ",np.sqrt(np.mean(mix_raw ** 2)))

q1  1256
q2  1072
q3  66
q4  680
q5  304
q6  12.590789473684211
q7  24
q8  0
q9  7
q11  0.102124155
